# 01 LTA collector: set up, start, monitor (project 1.3)

Controls `analysis/lta_collector.py`, which runs 24/7 in the background. This notebook never collects data itself.

**First time:** run cells 1 → 4 in order. **Daily:** run cell 1 and cell 5. Python kernel; working directory = project folder.

How it stays up: a Windows scheduled task starts the collector at log-on and every 10 minutes. If the collector is already running, the new copy exits immediately (file lock); if it crashed, or the PC rebooted, it is restarted within 10 minutes. The task has no 3-day time limit and keeps running on battery.

Keep the PC **on and awake** (Settings → System → Power: never sleep when plugged in). A sleeping PC collects nothing.

AI disclosure: drafted with Claude (Anthropic), 14 Oct 2026; log in `AI_USE.md`.

## 1. Paths and checks

In [ ]:
import json, subprocess, sys, time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from xml.sax.saxutils import escape

PROJ = Path.cwd()
SCRIPT = PROJ / "analysis" / "lta_collector.py"
RAW = PROJ / "data_raw"
TASK = "SLDS_LTA_collector_247"
SGT = timezone(timedelta(hours=8))
PYW = Path(sys.executable).with_name("pythonw.exe")
RUNNER = PYW if PYW.exists() else Path(sys.executable)

def key_present() -> bool:
    renv = PROJ / ".Renviron"
    return renv.exists() and any(l.strip().startswith("LTA_ACCOUNT_KEY=") and len(l.split("=", 1)[1].strip()) > 5
                                 for l in renv.read_text(encoding="utf-8-sig").splitlines())

def run(args: list[str]) -> str:
    out = subprocess.run(args, capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()

print("Project folder:", PROJ)
print("Collector script found:", SCRIPT.exists())
print("Key in .Renviron:", key_present())
print("Python used by the task:", RUNNER)

## 2. Test: one pass of every feed (about 30 s)
Expect `speedband_snapshots: 1`, saved incident/VMS/travel-time files, and `last_error` empty (or a single feed error you can read).

In [ ]:
print(run([sys.executable, str(SCRIPT), "--once"])[-2500:])

## 3. Remove the old 5-minute task (if you created one earlier)
Only one collector should run. This deletes the earlier `SLDS_LTA_collector` task from `collect.py`/`01_collect.R`; an error saying it does not exist is fine.

In [ ]:
print(run(["schtasks", "/Delete", "/TN", "SLDS_LTA_collector", "/F"]))

## 4. Create the watchdog task and start collecting (run once)

In [ ]:
start = datetime.now().replace(second=0, microsecond=0).isoformat()
xml = f"""<?xml version="1.0" encoding="UTF-16"?>
<Task version="1.2" xmlns="http://schemas.microsoft.com/windows/2004/02/mit/task">
  <RegistrationInfo><Description>SLDS 42.510 project 1.3: keeps lta_collector.py running</Description></RegistrationInfo>
  <Triggers>
    <LogonTrigger><Enabled>true</Enabled></LogonTrigger>
    <TimeTrigger>
      <Repetition><Interval>PT10M</Interval><StopAtDurationEnd>false</StopAtDurationEnd></Repetition>
      <StartBoundary>{start}</StartBoundary><Enabled>true</Enabled>
    </TimeTrigger>
  </Triggers>
  <Principals><Principal id="Author"><LogonType>InteractiveToken</LogonType><RunLevel>LeastPrivilege</RunLevel></Principal></Principals>
  <Settings>
    <MultipleInstancesPolicy>IgnoreNew</MultipleInstancesPolicy>
    <DisallowStartIfOnBatteries>false</DisallowStartIfOnBatteries>
    <StopIfGoingOnBatteries>false</StopIfGoingOnBatteries>
    <AllowHardTerminate>true</AllowHardTerminate>
    <StartWhenAvailable>true</StartWhenAvailable>
    <RunOnlyIfNetworkAvailable>false</RunOnlyIfNetworkAvailable>
    <IdleSettings><StopOnIdleEnd>false</StopOnIdleEnd><RestartOnIdle>false</RestartOnIdle></IdleSettings>
    <AllowStartOnDemand>true</AllowStartOnDemand>
    <Enabled>true</Enabled>
    <Hidden>false</Hidden>
    <RunOnlyIfIdle>false</RunOnlyIfIdle>
    <WakeToRun>false</WakeToRun>
    <ExecutionTimeLimit>PT0S</ExecutionTimeLimit>
    <Priority>7</Priority>
  </Settings>
  <Actions Context="Author">
    <Exec>
      <Command>{escape(str(RUNNER))}</Command>
      <Arguments>"{escape(str(SCRIPT))}"</Arguments>
      <WorkingDirectory>{escape(str(PROJ))}</WorkingDirectory>
    </Exec>
  </Actions>
</Task>
"""
xml_path = RAW / "collector_task.xml"
RAW.mkdir(exist_ok=True)
xml_path.write_text(xml, encoding="utf-16")
print(run(["schtasks", "/Create", "/TN", TASK, "/XML", str(xml_path), "/F"]))
print(run(["schtasks", "/Run", "/TN", TASK]))      # start now instead of waiting up to 10 minutes
time.sleep(20)
print(run(["schtasks", "/Query", "/TN", TASK, "/FO", "LIST"]))

## 5. Health check (run daily)
- **Heartbeat** under 2 minutes old = running.
- **Speed-band snapshots**: about 12 per hour, 288 per day; no gap over 10 minutes.
- **Failures**: a few retried failures are normal; many in a row = network or key problem (see `last_error` and the log).

In [ ]:
status = json.loads((RAW / "collector_status.json").read_text(encoding="utf-8"))
beat = datetime.strptime(status["last_heartbeat"], "%Y-%m-%dT%H:%M:%S%z")
age = (datetime.now(SGT) - beat).total_seconds() / 60
print(f"Heartbeat: {age:.1f} min ago -> {'RUNNING' if age < 2 else 'NOT RUNNING (check the task / PC sleep)'}")
print("Started:", status.get("started"), "| PID:", status.get("pid"))
print("Latest LTA update saved:", status.get("sb_last_updated"))
print("Today:", status.get("today"))
print("Last error:", status.get("last_error"))

import csv
rows = list(csv.DictReader(open(RAW / "log_speedbands.csv", encoding="utf-8")))
recent = [r for r in rows if datetime.strptime(r["retrieved_at"], "%Y-%m-%dT%H:%M:%S%z") > datetime.now(SGT) - timedelta(hours=24)]
ok = sorted(datetime.strptime(r["retrieved_at"], "%Y-%m-%dT%H:%M:%S%z") for r in recent if r["status"].startswith("ok"))
gaps = [(b - a).total_seconds() / 60 for a, b in zip(ok, ok[1:])]
print(f"\nLast 24 h: {len(ok)} snapshots saved (target ~288), {sum(not r['status'].startswith('ok') for r in recent)} failed fetches")
print(f"Longest gap between snapshots: {max(gaps):.1f} min" if gaps else "Not enough snapshots yet for gaps")
if ok:
    secs = [float(r["seconds"]) for r in recent if r["status"].startswith("ok")]
    print(f"Fetch time per snapshot: median {sorted(secs)[len(secs)//2]:.1f} s, max {max(secs):.1f} s")

feeds = list(csv.DictReader(open(RAW / "log_feeds.csv", encoding="utf-8")))
recent_f = [r for r in feeds if datetime.strptime(r["retrieved_at"], "%Y-%m-%dT%H:%M:%S%z") > datetime.now(SGT) - timedelta(hours=24)]
for name in sorted({r["feed"] for r in recent_f}):
    sub = [r for r in recent_f if r["feed"] == name]
    print(f"{name:14s} polls {len(sub):4d}  failed {sum(r['status'] != 'ok' for r in sub):3d}  saved {sum(r['saved'] == 'TRUE' for r in sub):4d}")

size_gb = sum(f.stat().st_size for f in RAW.rglob("*") if f.is_file()) / 1e9
days = max(1, (datetime.now().date() - datetime(2026, 10, 13).date()).days)
left = (datetime(2026, 11, 27).date() - datetime.now().date()).days
print(f"\ndata_raw: {size_gb:.2f} GB; at this rate about {size_gb / days * (days + left):.1f} GB by 27 Nov")
print("\nLast log lines:")
log_file = RAW / "logs" / "collector.log"
print("\n".join(log_file.read_text(encoding="utf-8").splitlines()[-8:]) if log_file.exists() else "no log yet")

## 6. Stop collecting (only when the team agrees, planned after Fri 27 Nov)
Uncomment and run. `/End` stops the running collector; `/Delete` removes the watchdog so it does not restart.

In [ ]:
# print(run(["schtasks", "/End", "/TN", TASK]))
# print(run(["schtasks", "/Delete", "/TN", TASK, "/F"]))